# Stage 5 - Baseline Evaluation

Goal: run the baseline Socratic agent on held-out test problems, then label each reply by hand.

Labels: 1 = good Socratic, 2 = too vague, 3 = gives answer too soon, 4 = other.

How to use: run the cells from top to bottom. The model name comes from `.env` (GEMINI_MODEL).
Results are saved after every example, so if the API is busy or the quota ends you can run the cell again later and it continues where it stopped.

In [2]:
import sys, json, time, random
from pathlib import Path
from collections import Counter

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from backend.app.models.socratic_agent import socratic_response, MODEL_NAME

TEST_PATH = ROOT / 'data' / 'processed' / 'socrateach_test.jsonl'
RESULTS_PATH = ROOT / 'experiments' / 'phase1_baseline' / 'baseline_eval_results.json'
NOTES_PATH = ROOT / 'experiments' / 'phase1_baseline' / 'baseline_eval_notes.md'

NUM_EXAMPLES = 15
SEED = 42
PAUSE_SECONDS = 5

print('Model being evaluated:', MODEL_NAME)

Model being evaluated: gemini-3.6-flash


## 1. Pick the examples
15 test records, each from a different problem, never the closing (END) turn. Fixed seed, so everyone gets the same 15.

In [3]:
records = []
with open(TEST_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        rec = json.loads(line)
        if not rec['metadata']['is_end']:
            records.append(rec)

random.seed(SEED)
random.shuffle(records)

examples = []
seen_problems = set()
for rec in records:
    pid = rec['metadata']['problem_id']
    if pid in seen_problems:
        continue
    seen_problems.add(pid)
    examples.append(rec)
    if len(examples) == NUM_EXAMPLES:
        break

print('Picked', len(examples), 'examples from different problems')

Picked 15 examples from different problems


## 2. Run the agent
The dataset stores the conversation as one text block, so we first turn it back into the list format the agent expects.
If the API is busy (503) the cell waits and retries. If it still fails, that example is skipped: just run the cell again later.

In [ ]:
def context_to_history(context):
    history = []
    for line in context.splitlines():
        if line.startswith('Teacher: '):
            history.append({'role': 'teacher', 'content': line[len('Teacher: '):]})
        elif line.startswith('Student: '):
            history.append({'role': 'student', 'content': line[len('Student: '):]})
        elif line.strip() and history:
            history[-1]['content'] += ' ' + line.strip()
    return history

ERROR_REPLY_START = 'I had trouble processing that turn'

RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
if RESULTS_PATH.exists():
    results = json.loads(RESULTS_PATH.read_text(encoding='utf-8'))
else:
    results = {}

def save_results():
    RESULTS_PATH.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding='utf-8')

for ex in examples:
    ex_id = ex['id']
    if ex_id in results and results[ex_id]['model'] == MODEL_NAME:
        continue
    history = context_to_history(ex['context'])
    reply = None
    for attempt in range(3):
        candidate = socratic_response(ex['problem'], history)
        if not candidate.startswith(ERROR_REPLY_START):
            reply = candidate
            break
        print('  API busy, waiting 20 seconds then retrying...')
        time.sleep(20)
    if reply is None:
        print('Skipped ' + ex_id + ' (API error). Run this cell again later.')
        continue
    results[ex_id] = {
        'model': MODEL_NAME,
        'problem': ex['problem'],
        'context': ex['context'],
        'agent_reply': reply,
        'reference_reply': ex['target'],
        'user_type': ex['metadata']['user_type'],
        'label': None,
    }
    save_results()
    print('Done: ' + ex_id)
    time.sleep(PAUSE_SECONDS)

print('Finished. Total saved:', len(results))

Done: GSM8K_train_6381_GSM8K_train_6381_4_turn0
Done: MAWPS_train_2002_MAWPS_train_2002_0_turn2
Done: GSM8K_test_1309_GSM8K_test_1309_1_turn0
Done: GSM8K_train_1448_GSM8K_train_1448_3_turn3
Done: MAWPS_train_1067_MAWPS_train_1067_2_turn2
Done: GSM8K_train_6105_GSM8K_train_6105_3_turn0
[socratic_agent] API call error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3.6-flash\nPlease retry in 6.355405572s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limi

## 3. Label each reply by hand
For each example you see the problem, the conversation so far, the agent's reply, and the real teacher's reply from the dataset for comparison.
Type 1, 2, 3 or 4 and press Enter. Type q to stop and continue later.

In [ ]:
OPTIONS = {'1': 'good Socratic', '2': 'too vague', '3': 'gives answer too soon', '4': 'other'}

for ex_id, r in results.items():
    if r['model'] != MODEL_NAME or r['label']:
        continue
    print('=' * 60)
    print('PROBLEM:', r['problem'])
    print()
    print('CONVERSATION SO FAR:')
    print(r['context'] if r['context'] else '(start of conversation)')
    print('AGENT REPLY:', r['agent_reply'])
    print()
    print('REFERENCE TEACHER (from dataset):', r['reference_reply'])
    print()
    choice = input('Label: 1=good Socratic, 2=too vague, 3=gives answer too soon, 4=other, q=stop: ').strip()
    if choice == 'q':
        break
    if choice in OPTIONS:
        r['label'] = OPTIONS[choice]
        save_results()

## 4. Tally and save the findings

In [ ]:
labelled = [r['label'] for r in results.values() if r['model'] == MODEL_NAME and r['label']]
counts = Counter(labelled)
total = len(labelled)

lines = ['# Baseline Evaluation Notes - Stage 5', '', 'Model: ' + MODEL_NAME, 'Examples labelled: ' + str(total), '']
for name in OPTIONS.values():
    n = counts.get(name, 0)
    pct = 100 * n / total if total else 0
    lines.append(f'- {name}: {n} ({pct:.0f}%)')

print('\n'.join(lines))
NOTES_PATH.write_text('\n'.join(lines) + '\n', encoding='utf-8')
print()
print('Saved to', NOTES_PATH)